# Essential HDFS Commands — Interactive Lab

This notebook teaches the essential HDFS commands covered in `docs/07-essential-hdfs-commands.md`.

Each section shows the **CLI command** syntax followed by its **equivalent Python code** via the HttpFS proxy — so you learn both the command and how to apply it programmatically.


## Setup

Connect to HDFS via the `proxy` gateway. No `subprocess` or `docker exec` needed — all operations go through HttpFS on port 14000.


In [ ]:
from hdfs import InsecureClient

client = InsecureClient('http://localhost:14000', user='root')
print('Connected to HDFS via proxy ✓')

## 1. Explore Directories

### CLI equivalent
```bash
hdfs dfs -ls /
hdfs dfs -ls -R /user
hdfs dfs -du -h /user
hdfs dfs -mkdir -p /lab-guide/data
```


In [ ]:
# List root directory
root_files = client.list('/')
print(f'Root: {root_files}')

# Create directories
client.makedirs('/lab-guide/data', permission=755)
client.makedirs('/lab-guide/backup', permission=755)
print('Directories created: /lab-guide/data, /lab-guide/backup')

# Check directory status
status = client.status('/lab-guide')
print(f"\n/lab-guide status: {status['length']} bytes, owner={status['owner']}, permission={status['permission']}")

## 2. Upload Data (Ingestion)

### CLI equivalent
```bash
hdfs dfs -put local_file.csv /raw_data/
```


In [ ]:
import sys; sys.path.insert(0, '../scripts')
import os
from hdfs_utils import download_stream

os.makedirs('../temp', exist_ok=True)
download_stream('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv', '../temp/sample_data.csv')

In [ ]:
# Upload to HDFS (equivalent to `hdfs dfs -put`)
client.upload(
    hdfs_path='/lab-guide/data/titanic.csv',
    local_path='../temp/sample_data.csv',
    overwrite=True,
    permission=644,
)
print('Uploaded: /lab-guide/data/titanic.csv ✓')

# Verify
files = client.list('/lab-guide/data')
print(f'Contents: {files}')

## 3. Download Data (Extraction)

### CLI equivalent
```bash
hdfs dfs -get /raw_data/file.csv ./my_file.csv
```


In [ ]:
# Download from HDFS to local (equivalent to `hdfs dfs -get`)
client.download(
    hdfs_path='/lab-guide/data/titanic.csv',
    local_path='../temp/downloaded_titanic.csv',
    overwrite=True,
)
print('Downloaded: ../temp/downloaded_titanic.csv ✓')

# Verify it was written
import os
size = os.path.getsize('../temp/downloaded_titanic.csv')
print(f'Local file size: {size} bytes')

## 4. Read File Contents

### CLI equivalent
```bash
hdfs dfs -cat /raw_data/sample.txt
hdfs dfs -tail /logs/system.log
```


In [ ]:
# Read entire file (equivalent to `hdfs dfs -cat`)
with client.read('/lab-guide/data/titanic.csv') as reader:
    content = reader.read()[:1000]  # first 1000 bytes
print('--- HEAD (first 1000 bytes) ---')
print(content.decode('utf-8'))

In [ ]:
# Read last 1KB (equivalent to `hdfs dfs -tail`)
status = client.status('/lab-guide/data/titanic.csv')
file_size = status['length']
chunk_size = min(1024, file_size)

with client.read('/lab-guide/data/titanic.csv', offset=file_size - chunk_size) as reader:
    tail = reader.read()
print(f'--- TAIL (last {chunk_size} bytes of {file_size} total) ---')
print(tail.decode('utf-8'))

## 5. Copy, Move, and Delete

### CLI equivalent
```bash
hdfs dfs -cp /raw_data/file.csv /backup/file.csv
hdfs dfs -mv /raw_data/old.csv /raw_data/new.csv
hdfs dfs -rm /raw_data/wrong_file.csv
hdfs dfs -rm -r /old_folder
```


In [ ]:
# Copy within HDFS (equivalent to `hdfs dfs -cp`)
client.rename('/lab-guide/data/titanic.csv', '/lab-guide/backup/titanic_backup.csv')
print('Copied (renamed) to: /lab-guide/backup/titanic_backup.csv ✓')
# Note: rename also works as move; for true copy via API, upload again

In [ ]:
# Upload again to original location (simulating cp + original retained)
client.upload('/lab-guide/data/titanic.csv', '../temp/sample_data.csv', overwrite=True)
print('Restored original: /lab-guide/data/titanic.csv ✓')

# List both locations
print(f"data/:   {client.list('/lab-guide/data')}")
print(f"backup/: {client.list('/lab-guide/backup')}")

In [ ]:
# Delete a file (equivalent to `hdfs dfs -rm`)
client.delete('/lab-guide/backup/titanic_backup.csv')
print('Deleted: /lab-guide/backup/titanic_backup.csv ✓')

# Verify
print(f"backup/ after delete: {client.list('/lab-guide/backup')}")

## 6. Permissions

### CLI equivalent
```bash
hdfs dfs -chmod 755 /raw_data
hdfs dfs -chown admin:users /raw_data
```


In [ ]:
# Change permissions (equivalent to `hdfs dfs -chmod`)
client.set_permission('/lab-guide/data', 700)
status = client.status('/lab-guide/data')
print(f"Updated permission: {status['permission']}")

# Change owner (equivalent to `hdfs dfs -chown`)
# Note: This requires superuser privileges (user='root' via proxy)
client.set_owner('/lab-guide/data', owner='root', group='supergroup')
status = client.status('/lab-guide/data')
print(f"Updated owner: {status['owner']}:{status['group']}")

## 7. Cluster Administration

### CLI equivalent
```bash
hdfs dfsadmin -report
hdfs dfsadmin -safemode get
```

These `dfsadmin` commands are not available via the `hdfs` Python library. To run them, enter the NameNode container:

```bash
make shell-namenode
# then inside: hdfs dfsadmin -report
```


In [ ]:
# We can check HDFS health via the client's status endpoint
try:
    root_status = client.status('/')
    print(f"HDFS root ✓ — {root_status['length']} bytes used")
except Exception as e:
    print(f"Could not reach HDFS: {e}")

print("\nTip: Run `make shell-namenode` and use `hdfs dfsadmin -report` for full cluster health.")

## 8. Blocks & Replication Factor (from Python)

`dfsadmin` needs the shell, but two things you *can* do straight from the proxy are **inspect where a file's blocks live** (WebHDFS `GETFILEBLOCKLOCATIONS`) and **change its replication factor** (`SETREPLICATION`).

### CLI equivalent
```bash
hdfs fsck /lab-guide/data/titanic.csv -files -blocks -locations
hdfs dfs -setrep -w 1 /lab-guide/data/titanic.csv
```

In [ ]:
import sys; sys.path.insert(0, '../scripts')
from hdfs_utils import block_report

FILE = '/lab-guide/data/titanic.csv'

In [ ]:
# Lower the replication factor to 1, then restore it to 3 — watch it change.
client.set_replication(FILE, 1)
print('Set replication = 1')
block_report(client, FILE)

client.set_replication(FILE, 3)
print('\nRestored replication = 3 (re-replication may take a few seconds)')
block_report(client, FILE)

The `replication` field in `status` flips immediately; the actual replica **deletion** (when lowering) or **creation** (when raising) is handled asynchronously by the NameNode, so re-run the report after a moment to see the host list settle. This is exactly how HDFS keeps every block at its target replication as nodes come and go.

## 9. Load HDFS Data into Pandas

In [ ]:
import pandas as pd
import io

# Read CSV from HDFS directly into a DataFrame
with client.read('/lab-guide/data/titanic.csv') as reader:
    df = pd.read_csv(io.StringIO(reader.read().decode('utf-8')))
print(f'Shape: {df.shape}')
df.head()

## 10. Cleanup

Remove the lab files from HDFS and the local temp directory.

In [ ]:
# Remove lab directory from HDFS
client.delete('/lab-guide', recursive=True)
print('Deleted: /lab-guide ✓')

# Clean local temp files
import os
for f in ['../temp/sample_data.csv', '../temp/downloaded_titanic.csv']:
    if os.path.exists(f):
        os.remove(f)
        print(f'Removed: {f}')

print('\nCleanup complete ✓')

## Summary

| CLI Command | Python Equivalent |
|---|---|
| `hdfs dfs -ls` | `client.list()` |
| `hdfs dfs -mkdir` | `client.makedirs()` |
| `hdfs dfs -du` | `client.status()` |
| `hdfs dfs -put` | `client.upload()` |
| `hdfs dfs -get` | `client.download()` |
| `hdfs dfs -cat` | `client.read()` |
| `hdfs dfs -cp` | `client.upload()` (re-upload) |
| `hdfs dfs -mv` | `client.rename()` |
| `hdfs dfs -rm` | `client.delete()` |
| `hdfs dfs -chmod` | `client.set_permission()` |
| `hdfs dfs -chown` | `client.set_owner()` |
| `hdfs dfsadmin` | `make shell-namenode` (docker exec) |
| `hdfs fsck -blocks -locations` | `GETFILEBLOCKLOCATIONS` via proxy (see §8) |
| `hdfs dfs -setrep` | `client.set_replication()` |
| `hdfs dfs -count` | `client.content()` |
